# 01d — Reasoning-effort runs (data generation only)

Run GPT-5.4 no-desc on `tuning_sample.csv` (the tuning set) at three values of
`reasoning_effort`: `low`, `medium` (default), and `high`.  Three API keys are
used so the three runs go in parallel, avoiding per-key rate limits.

**Outputs (saved to `data/results/llm/`):**
- `01d_predictions.csv` — one row per loan × variant: `actual`, `llm_pred`, `prob_fully_paid`, `reasoning_effort`, `llm_reasoning`
- `01d_metrics.csv` — accuracy / precision / recall / F1 / AUC per variant (default-threshold view)
- Per-call rows are also auto-appended to `llm_calls.csv`.

This notebook **only generates data**.  Threshold tuning, the medium-vs-high
decision, and the held-out validation all happen in `04_Final_Test_Analysis.ipynb`.

In [1]:
# llm_utils.py and llm_pricing.py live one directory up — make them importable.
import sys; sys.path.insert(0, "..")

import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv

from llm_utils import (
    load_llm_sample, run_ml_on_sample, run_llm_experiment,
    compare_results, evaluate_predictions, RESULTS_DIR,
)

In [2]:
# Load OpenAI API keys from .env — one key per effort level, no sharing.
load_dotenv("../.env", override=True)

from llm_utils import load_all_api_keys
_ALL_KEYS = load_all_api_keys("openai")
MAX_WORKERS = max(1, len(_ALL_KEYS) * 4)
print(f"Loaded {len(_ALL_KEYS)} API key(s) for shared use.")

Loaded 10 API key(s) for shared use.


In [3]:
# Tuning sample = the 100-loan LLM eval sample (built by 02_Preprocessing).
# Threshold tuning will use these labels in the next notebook.
tuning_sample = load_llm_sample()
y_true = tuning_sample['loan_status'].values
print(f"Tuning sample: {len(tuning_sample)} loans")
print(f"  Charged Off: {(y_true == 0).sum()}")
print(f"  Fully Paid:  {(y_true == 1).sum()}")

# XGBoost predictions on the same sample for context (printed only).
xgb_probs, xgb_preds = run_ml_on_sample(tuning_sample)
xgb_metrics = evaluate_predictions(y_true, xgb_preds.tolist(),
                                   label="XGBoost (tuning sample)",
                                   probabilities=xgb_probs.tolist())

Tuning sample: 100 loans
  Charged Off: 15
  Fully Paid:  85

XGBoost (tuning sample) Results (100 samples)
Accuracy: 69.0%
AUC:      0.660  (over 100 rows with logprobs)

Classification Report:
              precision    recall  f1-score   support

 Charged Off       0.23      0.47      0.31        15
  Fully Paid       0.89      0.73      0.80        85

    accuracy                           0.69       100
   macro avg       0.56      0.60      0.56       100
weighted avg       0.79      0.69      0.73       100

Confusion Matrix:
[[ 7  8]
 [23 62]]


In [ ]:
# Run three GPT-5.4 variants concurrently, each with its own API key.
FORCE_RERUN = False  # Set to True to re-run API calls from scratch (bypasses cache)
MODEL = "gpt-5.4"
EFFORTS = ["low", "medium", "high"]

def run_one(effort):
    return run_llm_experiment(
        tuning_sample,
        use_cache=not FORCE_RERUN,
        api_provider="openai",
        model_name=MODEL,
        api_keys=_ALL_KEYS,
        max_workers=MAX_WORKERS,
        label=f"GPT-5.4 reasoning={effort}",
        include_desc=False,
        with_logprobs=True,
        reasoning_effort=effort,
    )

results = {}
with ThreadPoolExecutor(max_workers=3) as ex:
    futures = {ex.submit(run_one, e): e for e in EFFORTS}
    for fut in futures:
        e = futures[fut]
        results[e] = fut.result()

print("\nAll three runs complete.")

In [ ]:
# Build a single consolidated predictions CSV (one row per loan × variant).
# Per-loan tokens/cost are embedded so cost is derivable from this file alone.
rows = []
for effort, res in results.items():
    ti, to, cu = res.get('input_tokens'), res.get('output_tokens'), res.get('cost_usd')
    for i in range(len(tuning_sample)):
        rows.append({
            "row_index":         i,
            "reasoning_effort":  effort,
            "actual":            int(y_true[i]),
            "llm_pred":          res['predictions'][i],
            "prob_fully_paid":   res['probabilities'][i],
            "llm_reasoning":     res['reasonings'][i],
            "xgb_pred":          int(xgb_preds[i]),
            "xgb_prob":          float(xgb_probs[i]),
            "input_tokens":      ti[i] if ti else None,
            "output_tokens":     to[i] if to else None,
            "cost_usd":          cu[i] if cu else None,
        })

predictions_df = pd.DataFrame(rows)
out_path = f"{RESULTS_DIR}/01d_predictions.csv"
predictions_df.to_csv(out_path, index=False)
print(f"Saved {len(predictions_df)} rows to {out_path}")

In [6]:
# Summary metrics per variant at the LLM's default (hard 0/1) prediction.
# These are the BEFORE-tuning numbers; tuned-threshold metrics are in 04_Final_Test_Analysis.ipynb.
metrics_rows = []
for effort, res in results.items():
    m = res['metrics'].copy()
    m['reasoning_effort'] = effort
    m['variant'] = f"GPT-5.4 reasoning={effort}"
    metrics_rows.append(m)

metrics_rows.append({**xgb_metrics, 'reasoning_effort': '-', 'variant': 'XGBoost (tuning)'})

metrics_df = pd.DataFrame(metrics_rows)
cols = ['variant', 'reasoning_effort', 'accuracy', 'auc',
        'precision_charged_off', 'recall_charged_off', 'f1_charged_off']
metrics_df = metrics_df[[c for c in cols if c in metrics_df.columns]]

out_path = f"{RESULTS_DIR}/01d_metrics.csv"
metrics_df.to_csv(out_path, index=False)
print(metrics_df.to_string(index=False))
print(f"\nSaved metrics to {out_path}")

               variant reasoning_effort  accuracy      auc  precision_charged_off  recall_charged_off  f1_charged_off
   GPT-5 reasoning=low              low      0.74      NaN               0.280000            0.466667        0.350000
GPT-5 reasoning=medium           medium      0.76      NaN               0.263158            0.333333        0.294118
  GPT-5 reasoning=high             high      0.80      NaN               0.352941            0.400000        0.375000
      XGBoost (tuning)                -      0.65 0.670588               0.205882            0.466667        0.285714

Saved metrics to /Users/alemz/Projects/Github/Sabadell_Capstone/data/results/llm/01d_metrics.csv


## High-effort consistency check (standalone)

`reasoning_effort="high"` is the best single-run performer above, so here we
test how **stable** it is by running it `N` times on the same `tuning_sample`
and measuring prediction flips — same stability definitions as `01b`.

**This block is independent of the low/med/high run above.** It depends only on
the setup cells (imports, API keys, `tuning_sample`, `y_true`), so you can
re-run just these two cells without re-touching the earlier experiment. Each
run uses a distinct label (`... | consistency run{n}`) so the cache and
`llm_calls.csv` never collide.

**Outputs (separate from the main `01d_*` files):**
- `01d_high_consistency_predictions.csv` — per-loan run matrix + `all_agree` / `majority_vote`
- `01d_high_consistency_metrics.csv` — per-run accuracy / precision / recall / F1 / AUC

In [4]:
# ── HIGH-EFFORT CONSISTENCY CHECK (standalone) ──────────────────────────────
# Self-contained: needs ONLY the setup cells above (imports, _ALL_KEYS,
# MAX_WORKERS, tuning_sample, y_true). It does NOT use `results` from the
# low/med/high run, so you can re-run just this block without re-running that.
#   → Run the 3 setup cells once, then run this cell as often as you like.
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed

N_CONSISTENCY_RUNS = 3              # same N as 01b for comparability
CONSISTENCY_FORCE_RERUN = False    # True = bypass cache and re-hit the API

def run_high(run_idx):
    # Distinct label per run so cached rows / llm_calls.csv don't collide.
    return run_llm_experiment(
        tuning_sample,
        use_cache=not CONSISTENCY_FORCE_RERUN,
        api_provider="openai",
        model_name="gpt-5.4",
        api_keys=_ALL_KEYS,
        max_workers=MAX_WORKERS,
        label=f"GPT-5.4 reasoning=high | consistency run{run_idx + 1}",
        include_desc=False,
        with_logprobs=True,
        reasoning_effort="high",
    )

high_runs = [None] * N_CONSISTENCY_RUNS
with ThreadPoolExecutor(max_workers=N_CONSISTENCY_RUNS) as ex:
    futures = {ex.submit(run_high, i): i for i in range(N_CONSISTENCY_RUNS)}
    for fut in as_completed(futures):
        high_runs[futures[fut]] = fut.result()

# ── Stability across the N runs (same definitions as 01b) ───────────────────
pred_cols = pd.DataFrame({f'run_{i+1}': r['predictions'] for i, r in enumerate(high_runs)})
all_agree = pred_cols.nunique(axis=1) == 1
n_stable = int(all_agree.sum())
majority_vote = pred_cols.mode(axis=1)[0].astype(int)
majority_acc = (majority_vote.values == y_true).mean()

accs = np.array([r['metrics']['accuracy'] for r in high_runs])
f1s  = np.array([r['metrics']['f1_charged_off'] for r in high_runs])

print(f"\n{'='*60}\nGPT-5.4 reasoning=high — consistency over {N_CONSISTENCY_RUNS} runs\n{'='*60}")
print(f"  Per-loan stable (all runs agree): {n_stable}/{len(tuning_sample)} ({n_stable*100/len(tuning_sample):.0f}%)")
print(f"  Majority-vote accuracy:           {majority_acc*100:.1f}%")
print(f"  Accuracy: mean {accs.mean()*100:.1f}%  range {accs.min()*100:.1f}-{accs.max()*100:.1f}%  std {accs.std()*100:.2f}pp")
print(f"  CO F1:    mean {f1s.mean():.3f}  range {f1s.min():.3f}-{f1s.max():.3f}  std {f1s.std():.3f}")

/Users/alemz/Projects/Github/Sabadell_Capstone/notebooks/llm_models/01_model_selection/../llm_utils.py:1401: UserWarning: OpenAI model gpt-5.4 does not support logprobs under current settings. Falling back to calling without logprobs.
  return call_llm(


[GPT-5.4 reasoning=high | consistency run3 | no_desc] First call OK (pred=1, prob=None). Running 100 loans on 10 key(s) × 40 worker(s)...
[GPT-5.4 reasoning=high | consistency run1 | no_desc] First call OK (pred=1, prob=None). Running 100 loans on 10 key(s) × 40 worker(s)...
[GPT-5.4 reasoning=high | consistency run2 | no_desc] First call OK (pred=1, prob=None). Running 100 loans on 10 key(s) × 40 worker(s)...
[GPT-5.4 reasoning=high | consistency run3 | no_desc] 26/100 done (7s elapsed, ~22s remaining)
[GPT-5.4 reasoning=high | consistency run1 | no_desc] 26/100 done (7s elapsed, ~20s remaining)
[GPT-5.4 reasoning=high | consistency run2 | no_desc] 26/100 done (6s elapsed, ~19s remaining)
[GPT-5.4 reasoning=high | consistency run3 | no_desc] 51/100 done (12s elapsed, ~12s remaining)
[GPT-5.4 reasoning=high | consistency run2 | no_desc] 51/100 done (11s elapsed, ~10s remaining)
[GPT-5.4 reasoning=high | consistency run1 | no_desc] 51/100 done (11s elapsed, ~11s remaining)
[GPT-5.4 reas

In [5]:
# Export the high-effort consistency results to their OWN files so the main
# 01d_predictions.csv / 01d_metrics.csv (the low/med/high single runs) stay intact.
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

# Per-loan prediction matrix (one row per loan, one column per run) + stability flags.
pred_matrix = pd.DataFrame({f'run_{i+1}': r['predictions'] for i, r in enumerate(high_runs)})
pred_matrix.insert(0, 'row_index', range(len(tuning_sample)))
pred_matrix['actual'] = y_true

run_cols = [c for c in pred_matrix.columns if c.startswith('run_')]
pred_matrix['all_agree'] = pred_matrix[run_cols].nunique(axis=1) == 1
pred_matrix['majority_vote'] = pred_matrix[run_cols].mode(axis=1)[0].astype(int)
pred_matrix['majority_correct'] = (pred_matrix['majority_vote'] == pred_matrix['actual']).astype(int)

# Embed per-run, per-loan cost (named cost_usd_run_* so it isn't read as a prediction column).
for i, r in enumerate(high_runs):
    cu = r.get('cost_usd')
    if cu is not None:
        pred_matrix[f'cost_usd_run_{i+1}'] = cu

pred_path = f"{RESULTS_DIR}/01d_high_consistency_predictions.csv"
pred_matrix.to_csv(pred_path, index=False)

# Per-run metrics (so accuracy/F1 spread across runs is recoverable from the file).
metrics_rows = []
for i, r in enumerate(high_runs):
    m = r['metrics'].copy()
    m['run'] = i + 1
    m['reasoning_effort'] = 'high'
    metrics_rows.append(m)
metrics_df = pd.DataFrame(metrics_rows)
metrics_path = f"{RESULTS_DIR}/01d_high_consistency_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print(f"Saved {len(pred_matrix)} loan rows to {pred_path}")
print(f"Saved {len(metrics_df)} per-run metric rows to {metrics_path}")

Saved 100 loan rows to /Users/alemz/Projects/Github/Sabadell_Capstone/data/results/llm/01d_high_consistency_predictions.csv
Saved 3 per-run metric rows to /Users/alemz/Projects/Github/Sabadell_Capstone/data/results/llm/01d_high_consistency_metrics.csv
